# ML Feature Importance — Residential ISP v2 (XGBoost)

Reproduces two plots from `new_ml_tagging_final_access.ipynb` using the model saved by
`ml_tagging_example.ipynb` (`AssignMLTag` with `model_dir=./data/residential_isp_v2`).

1. **Top 10 features by SHAP** (mean |SHAP|)
2. **Max prevalence** by Censys port/service/OS family (highlight |Δ| ≥ 10%)

## Step 1: Load snapshot, labels, and saved model

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from as_tagging import ASTagging, OfflineSnapshotProvider, normalize_asn_input
from as_tagging.ml import MLTagger
from as_tagging.ml.feature_engineering import apply_cat_schema

DATA_PATH = "../as_feature_zenodo"  # same as ml_tagging_example.ipynb
DATE = "2026-01"
MODEL_DIR = "./data/residential_isp_v2"

offline_provider = OfflineSnapshotProvider(DATA_PATH)
tagger = ASTagging(snapshot_provider=offline_provider, date=DATE, use_cache=True)
print(f"Loaded {len(tagger.atomic_tags)} ASNs")

with open("./data/residential_isp/all_residential_eyeball_asns.json") as f:
    positive_asns = json.load(f)["asns"]
with open("./data/residential_isp/all_non_residential_eyeball_asns.json") as f:
    negative_asns = json.load(f)["asns"]

# Scope to eyeball ASes (same filter as ml_tagging_example.ipynb)
eyeball_asns = tagger.ListASNsWithoutTag("No Eyeball", treat_false_as_missing=True)
eyeball_canon = set(normalize_asn_input(a) for a in eyeball_asns)
positive_asns = [a for a in positive_asns if normalize_asn_input(a) in eyeball_canon]
negative_asns = [a for a in negative_asns if normalize_asn_input(a) in eyeball_canon]
print(f"Labels: {len(positive_asns)} positive, {len(negative_asns)} negative (eyeball scope)")

if not os.path.isdir(MODEL_DIR):
    raise FileNotFoundError(
        f"Saved model not found at {MODEL_DIR}.\n"
        "Run the 'Residential ISP v2' cell in ml_tagging_example.ipynb first "
        "(with model_dir=MODEL_DIR)."
    )

manifest = snapshot_schema = None
if hasattr(tagger.snapshot_provider, "get_manifest"):
    try:
        manifest = tagger.snapshot_provider.get_manifest(DATE)
    except Exception:
        pass
if hasattr(tagger.snapshot_provider, "get_schema"):
    try:
        snapshot_schema = tagger.snapshot_provider.get_schema(DATE)
    except Exception:
        pass

ml = MLTagger(
    snapshot_dict=tagger.atomic_tags,
    manifest=manifest,
    snapshot_schema=snapshot_schema,
    model_path=MODEL_DIR,
    verbose=True,
)
print(f"Loaded model: {ml._best_model_name}")

## Step 2: Build label vector and model input matrix

In [ ]:
def build_y_array(index, positive, negative):
    pos = {normalize_asn_input(a) for a in positive}
    neg = {normalize_asn_input(a) for a in negative}
    y = np.full(len(index), -1, dtype=np.int32)
    for i, asn in enumerate(index):
        canon = normalize_asn_input(asn)
        if canon in pos:
            y[i] = 1
        elif canon in neg:
            y[i] = 0
    return y

df_raw = ml._feature_df
y_all = build_y_array(df_raw.index, positive_asns, negative_asns)
labeled_mask = y_all >= 0
print(f"Labeled ASNs: {labeled_mask.sum()} ({(y_all==1).sum()} pos, {(y_all==0).sum()} neg)")

# Encoded design matrix (same columns/order as training)
df_encoded = apply_cat_schema(df_raw, ml._cat_cols, ml._cat_schema)
feature_cols = ml._num_cols + ml._cat_cols
X_explain = df_encoded[feature_cols].copy()
print(f"X_explain shape: {X_explain.shape}")

## Graph 1: Top 10 features by SHAP

Uses XGBoost-native SHAP contributions (`pred_contribs=True`) on labeled ASNs.

In [ ]:
import xgboost as xgb

if ml._best_model_name != "xgboost":
    raise RuntimeError(f"Expected xgboost model, got {ml._best_model_name}")

booster = ml._best_model.clf.get_booster()

# SHAP on labeled ASNs only (training set)
X_shap = X_explain.loc[labeled_mask]

dtest = xgb.DMatrix(
    data=X_shap,
    feature_names=list(X_shap.columns),
    enable_categorical=True,
)
contribs = booster.predict(dtest, pred_contribs=True)
shap_vals = contribs[:, :-1]

global_imp = np.abs(shap_vals).mean(axis=0)
feat_rank = pd.Series(global_imp, index=X_shap.columns).sort_values(ascending=False)
print(feat_rank.head(15))

In [ ]:
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
})

mean_abs = np.abs(shap_vals).mean(axis=0)
feat_imp = pd.DataFrame({
    "feature": X_shap.columns,
    "mean_abs_shap": mean_abs,
})

def source_of(name: str) -> str:
    s = str(name)
    i = s.find("_")
    return s if i < 0 else s[:i]

feat_imp["source"] = feat_imp["feature"].map(source_of)
src_imp = (
    feat_imp.groupby("source", as_index=False)["mean_abs_shap"]
    .sum()
    .sort_values("mean_abs_shap", ascending=False)
)

TOPN = 10
feat_top = feat_imp.sort_values("mean_abs_shap", ascending=False).head(TOPN).copy()
src_top = src_imp.head(TOPN).copy()

sources_union = pd.Index(feat_top["source"]).union(src_top["source"])
cmap = plt.get_cmap("tab20")
src2color = {s: cmap(i % cmap.N) for i, s in enumerate(sources_union)}
feat_top["color"] = feat_top["source"].map(src2color)
src_top["color"] = src_top["source"].map(src2color)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 6), gridspec_kw={"wspace": 0.1})

ft = feat_top.iloc[::-1]
barsL = axL.barh(ft["feature"], ft["mean_abs_shap"], color=ft["color"])
axL.set_title("Top 10 features by SHAP")
axL.set_xlabel("Mean |SHAP|")
axL.grid(axis="x", ls=":", alpha=0.35)
axL.bar_label(barsL, labels=[f"{v:.2f}" for v in ft["mean_abs_shap"]], padding=3, fontsize=12)
for spine in ["top", "right"]:
    axL.spines[spine].set_visible(False)

st = src_top.iloc[::-1]
barsR = axR.barh(st["source"], st["mean_abs_shap"], color=st["color"])
axR.set_title("Top 10 sources (sum of mean |SHAP|)")
axR.set_xlabel("Sum of mean |SHAP|")
axR.yaxis.tick_right()
axR.yaxis.set_label_position("right")
axR.invert_xaxis()
axR.grid(axis="x", ls=":", alpha=0.35)
axR.bar_label(barsR, labels=[f"{v:.2f}" for v in st["mean_abs_shap"]], padding=3, fontsize=12)
for spine in ["top", "left"]:
    axR.spines[spine].set_visible(False)

axL.tick_params(axis="both", which="major", labelsize=12)
axR.tick_params(axis="both", which="major", labelsize=12)
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

## Graph 2: Censys family prevalence (presence + max prevalence)

Port/service/OS families mapped from Censys top-k name/frac columns.
Uses raw (pre-encoding) feature values from the saved model pipeline.

In [ ]:
# --- Family mappers (from new_ml_tagging_final_access.ipynb) ---
PORT_TO_FAMILY = {
    80: "web", 443: "web", 8080: "web", 8443: "web", 8000: "web", 8081: "web",
    53: "dns", 853: "dns", 5353: "dns",
    25: "mail", 465: "mail", 587: "mail", 2525: "mail",
    110: "mail", 995: "mail", 143: "mail", 993: "mail",
    22: "remote_access", 23: "remote_access", 3389: "remote_access",
    5900: "remote_access", 5985: "remote_access", 5986: "remote_access", 5938: "remote_access",
    20: "file_transfer", 21: "file_transfer", 989: "file_transfer", 990: "file_transfer",
    69: "file_transfer", 2049: "file_transfer", 873: "file_transfer",
    389: "directory_auth", 636: "directory_auth", 88: "directory_auth",
    1812: "directory_auth", 1813: "directory_auth",
    445: "ms_fileshare", 135: "ms_fileshare", 137: "ms_fileshare",
    138: "ms_fileshare", 139: "ms_fileshare",
    1433: "database", 1521: "database", 3306: "database", 5432: "database",
    27017: "database", 6379: "database", 11211: "database", 9200: "database",
    2000: "voip_rtp", 5060: "voip_rtp", 5061: "voip_rtp", 1720: "voip_rtp", 58000: "voip_rtp",
    554: "streaming",
    1194: "vpn_tunnel", 1701: "vpn_tunnel", 500: "vpn_tunnel", 4500: "vpn_tunnel",
    1723: "vpn_tunnel", 51820: "vpn_tunnel",
    123: "time_sync",
    502: "ics_iot", 102: "ics_iot", 20000: "ics_iot", 47808: "ics_iot",
    161: "network_mgmt", 162: "network_mgmt", 514: "network_mgmt",
    67: "network_mgmt", 68: "network_mgmt", 546: "network_mgmt", 547: "network_mgmt",
    179: "network_mgmt", 89: "network_mgmt", 520: "network_mgmt", 646: "network_mgmt",
    7547: "network_mgmt", 30005: "network_mgmt", 2048: "network_mgmt", 8089: "network_mgmt",
}
RANGE_TO_FAMILY = [
    (range(16384, 32768), "voip_rtp"),
    (range(6881, 6890), "p2p"),
]

def port_family(x):
    try:
        p = int(str(x).strip())
    except Exception:
        return "other"
    fam = PORT_TO_FAMILY.get(p)
    if fam:
        return fam
    for rng, fam in RANGE_TO_FAMILY:
        if p in rng:
            return fam
    return "other"

def service_family(s):
    s = str(s).lower()
    if "http" in s: return "http_family"
    if "tls" in s or "ssl" in s: return "http_family"
    if "ssh" in s: return "ssh"
    if "smtp" in s: return "smtp"
    if "dns" in s: return "dns"
    if "sip" in s: return "sip"
    if "snmp" in s: return "snmp"
    if "telnet" in s: return "telnet"
    if "rdp" in s: return "rdp"
    if "cwmp" in s or "tr-069" in s or "tr069" in s: return "tr069"
    if "ipsec" in s or "isakmp" in s or "ike" in s: return "ipsec"
    return "other"

def os_family(s):
    s = str(s).lower()
    if "windows" in s: return "windows"
    if "linux" in s: return "linux"
    if any(k in s for k in ["routeros", "fortios", "vyos", "openwrt", "ios-xe", "ios xr", "junos", "huawei vrp", "mikrotik"]):
        return "network_os"
    if any(k in s for k in ["freebsd", "openbsd"]): return "bsd"
    if any(k in s for k in ["android", "ios "]): return "mobile"
    return "other"


def _long_topk(df, kind, family_mapper, ks=(1, 2, 3), include_other=True):
    parts = []
    for k in ks:
        name_col = f"censys_{kind}_{k}_name"
        frac_col = f"censys_{kind}_{k}_frac"
        if name_col not in df.columns or frac_col not in df.columns:
            continue
        fam = df[name_col].map(family_mapper).fillna("other")
        frac = pd.to_numeric(df[frac_col], errors="coerce").fillna(0.0)
        parts.append(pd.DataFrame({"idx": df.index, "family": fam.values, "frac": frac.values}))
    if not parts:
        return pd.DataFrame(columns=["idx", "family", "frac"])
    long = pd.concat(parts, ignore_index=True)
    if not include_other:
        long = long[long["family"] != "other"]
    return long


def _per_family_presence_and_max(long_df, y):
    if long_df.empty:
        empty_pres = pd.DataFrame(columns=[
            "family", "ratio_access", "ratio_nonaccess",
            "n_access", "n_nonaccess",
        ])
        empty_mm = pd.DataFrame(columns=[
            "family", "mean_max_access", "mean_max_nonaccess",
            "present_access", "present_nonaccess",
        ])
        return empty_pres, empty_mm

    agg = long_df.groupby(["idx", "family"], as_index=False).agg(max_frac=("frac", "max"))
    agg["present"] = 1
    lab = y.rename("label").to_frame()
    agg = agg.merge(lab, left_on="idx", right_index=True, how="inner")
    agg = agg[agg["label"].isin([0, 1])]

    n_access = int((lab["label"] == 1).sum())
    n_nonaccess = int((lab["label"] == 0).sum())

    pres_rows = []
    for fam, sub in agg.groupby("family"):
        n_acc = int(sub.loc[sub["label"] == 1, "idx"].nunique())
        n_non = int(sub.loc[sub["label"] == 0, "idx"].nunique())
        pres_rows.append((
            fam,
            n_acc / max(n_access, 1),
            n_non / max(n_nonaccess, 1),
            n_acc,
            n_non,
        ))
    presence_ratio = pd.DataFrame(pres_rows, columns=[
        "family", "ratio_access", "ratio_nonaccess", "n_access", "n_nonaccess",
    ]).sort_values(["ratio_access", "ratio_nonaccess"], ascending=False)

    mm_rows = []
    for fam, sub in agg.groupby("family"):
        sub_acc = sub.loc[sub["label"] == 1, "max_frac"]
        sub_non = sub.loc[sub["label"] == 0, "max_frac"]
        n_acc = int(sub_acc.shape[0])
        n_non = int(sub_non.shape[0])
        mm_rows.append((
            fam,
            float(sub_acc.mean()) if n_acc else np.nan,
            float(sub_non.mean()) if n_non else np.nan,
            n_acc,
            n_non,
        ))
    mean_max_when_present = pd.DataFrame(mm_rows, columns=[
        "family", "mean_max_access", "mean_max_nonaccess", "present_access", "present_nonaccess",
    ]).sort_values(["mean_max_access", "mean_max_nonaccess"], ascending=False)

    return presence_ratio, mean_max_when_present


def compute_family_metrics(df, y, include_other=False, ks=(1, 2, 3)):
    mask = pd.Series(y).values >= 0
    dfL = df.loc[mask].copy()
    yL = pd.Series(y, index=df.index)[mask].astype(int)
    out = {}
    for kind, mapper in [("port", port_family), ("service", service_family), ("os", os_family)]:
        long_df = _long_topk(dfL, kind, mapper, ks=ks, include_other=include_other)
        out[kind] = _per_family_presence_and_max(long_df, yL)
    return out

metrics = compute_family_metrics(df_raw, y_all, include_other=False)
port_presence, port_meanmax = metrics["port"]
print("[PORT] top presence ratios:\n", port_presence.head(10))

In [ ]:
port_presence, _ = metrics["port"]
srv_presence, _ = metrics["service"]
os_presence, _ = metrics["os"]
_, port_meanmax = metrics["port"]
_, srv_meanmax = metrics["service"]
_, os_meanmax = metrics["os"]

def _slim_presence(df, group):
    return df[["family", "ratio_access", "ratio_nonaccess"]].assign(group=group)

def _slim_meanmax(df, group):
    return df[["family", "mean_max_access", "mean_max_nonaccess"]].assign(group=group)

def pretty_label(group, fam):
    return f"{group} " + fam.replace("_", " ")

TP = pd.concat([
    _slim_presence(port_presence, "port"),
    _slim_presence(srv_presence, "srv"),
    _slim_presence(os_presence, "os"),
], ignore_index=True)
TM = pd.concat([
    _slim_meanmax(port_meanmax, "port"),
    _slim_meanmax(srv_meanmax, "srv"),
    _slim_meanmax(os_meanmax, "os"),
], ignore_index=True)
T = TP.merge(TM, on=["family", "group"], how="left")

HIGHLIGHT_PRES_THRESH = 0.10
HIGHLIGHT_MAXP_THRESH = 0.10
TOPN = 25

T["delta_presence"] = T["ratio_access"] - T["ratio_nonaccess"]
T["delta_maxprev"] = T["mean_max_access"] - T["mean_max_nonaccess"]
T["label"] = [pretty_label(g, f) for g, f in zip(T["group"], T["family"])]
T = T.sort_values(["delta_presence", "ratio_access"], ascending=[False, False]).reset_index(drop=True)
Tplot = T.head(TOPN).copy()

Tplot["hi_left"] = Tplot["delta_presence"].abs() >= HIGHLIGHT_PRES_THRESH
Tplot["hi_right"] = Tplot["delta_maxprev"].abs() >= HIGHLIGHT_MAXP_THRESH
Tplot["hi_any"] = Tplot["hi_left"] | Tplot["hi_right"]
Tplot["plot_mean_max_access"] = Tplot["mean_max_access"].clip(upper=1)
Tplot["plot_mean_max_nonaccess"] = Tplot["mean_max_nonaccess"].clip(upper=1)

COLOR_NON = "#1f77b4"
COLOR_ACC = "#2ca02c"

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 7), sharey=True, gridspec_kw={"wspace": 0})
plt.subplots_adjust(wspace=0)
y = np.arange(len(Tplot))[::-1]

# LEFT: presence ratio
for i, r in enumerate(Tplot.itertuples(index=False)):
    if r.hi_left:
        axL.axhspan(y[i] - 0.5, y[i] + 0.5, color="#000", alpha=0.06, lw=0)
for i, r in enumerate(Tplot.itertuples(index=False)):
    color = COLOR_ACC if r.delta_presence >= 0 else COLOR_NON
    axL.hlines(y[i], xmin=r.ratio_nonaccess, xmax=r.ratio_access, color=color, lw=2, alpha=0.9)
axL.scatter(Tplot["ratio_nonaccess"], y, s=40, color=COLOR_NON, label="non-residential")
axL.scatter(Tplot["ratio_access"], y, s=40, color=COLOR_ACC, label="residential")
for i, r in enumerate(Tplot.itertuples(index=False)):
    leftmost = min(r.ratio_nonaccess, r.ratio_access)
    rightmost = max(r.ratio_nonaccess, r.ratio_access)
    if rightmost < 0.8:
        x_text, ha = rightmost + 0.01, "left"
    else:
        x_text, ha = leftmost - 0.01, "right"
    axL.text(x_text, y[i], f"Δ={r.delta_presence:+.3f}", va="center", ha=ha, fontsize=8, color="#555")
axL.set_yticks(y)
axL.set_yticklabels([
    (lab if not hi else r"$\bf{" + lab.replace(" ", r"\ ") + "}$")
    for lab, hi in zip(Tplot["label"], Tplot["hi_any"])
])
axL.tick_params(axis="y", which="both", labelleft=True, length=3)
axL.set_xlim(0, 1)
axL.set_ylim(-0.5, len(Tplot) - 0.5)
axL.grid(axis="x", ls=":", alpha=0.4)
axL.set_title(f"Presence ratio (highlight |Δ|≥{int(HIGHLIGHT_PRES_THRESH * 100)}%)")
axL.set_xlabel("")
ticksL = axL.get_xticks()
axL.tick_params(axis="both", which="major", labelsize=10)
axL.set_xticks(ticksL)
axL.set_xticklabels(["" if np.isclose(t, axL.get_xlim()[1]) else f"{t:.1f}" for t in ticksL])

# RIGHT: max prevalence when present
for i, r in enumerate(Tplot.itertuples(index=False)):
    if r.hi_right:
        axR.axhspan(y[i] - 0.5, y[i] + 0.5, color="#000", alpha=0.06, lw=0)
for i, r in enumerate(Tplot.itertuples(index=False)):
    if np.isnan(r.plot_mean_max_nonaccess) or np.isnan(r.plot_mean_max_access):
        continue
    color = COLOR_ACC if (not np.isnan(r.delta_maxprev) and r.delta_maxprev >= 0) else COLOR_NON
    axR.hlines(y[i], xmin=r.plot_mean_max_nonaccess, xmax=r.plot_mean_max_access, color=color, lw=2, alpha=0.9)
mask_non = ~Tplot["plot_mean_max_nonaccess"].isna()
mask_acc = ~Tplot["plot_mean_max_access"].isna()
axR.scatter(Tplot.loc[mask_non, "plot_mean_max_nonaccess"], y[mask_non], s=40, color=COLOR_NON)
axR.scatter(Tplot.loc[mask_acc, "plot_mean_max_access"], y[mask_acc], s=40, color=COLOR_ACC)
for i, r in enumerate(Tplot.itertuples(index=False)):
    if np.isnan(r.delta_maxprev):
        continue
    leftmost_plot = np.nanmin([r.plot_mean_max_nonaccess, r.plot_mean_max_access])
    rightmost_plot = np.nanmax([r.plot_mean_max_nonaccess, r.plot_mean_max_access])
    if np.isnan(leftmost_plot) or np.isnan(rightmost_plot):
        continue
    rightmost_plot = min(rightmost_plot, 1.0)
    if rightmost_plot < 0.8:
        x_text, ha = min(rightmost_plot + 0.01, 0.99), "left"
    else:
        x_text, ha = leftmost_plot - 0.01, "right"
    axR.text(x_text, y[i], f"Δ={r.delta_maxprev:+.3f}", va="center", ha=ha, fontsize=8, color="#555")
axR.set_yticks(y)
axR.tick_params(axis="y", which="both", labelleft=False, length=3)
axR.set_xlim(0, 1)
ticksR = axR.get_xticks()
axR.tick_params(axis="both", which="major", labelsize=10)
axR.set_xticks(ticksR)
axR.set_xticklabels(["" if np.isclose(t, axR.get_xlim()[1]) else f"{t:.1f}" for t in ticksR])
axR.grid(axis="x", ls=":", alpha=0.4)
axR.set_title(f"Max prevalence (highlight |Δ|≥{int(HIGHLIGHT_MAXP_THRESH * 100)}%)")
axR.set_xlabel("")

plt.subplots_adjust(wspace=0, left=0.26, bottom=0.16)
handles = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor=COLOR_NON, markeredgecolor="none", markersize=12, label="non-residential"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor=COLOR_ACC, markeredgecolor="none", markersize=12, label="residential"),
]
fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.05))
plt.show()